# Modélisation : effet des médailles olympiques sur les licenciés

## Objectif
On cherche à mesurer si les médailles obtenues aux JO (par sport) sont associées à une augmentation du nombre de licenciés les années suivantes.

On utilise deux approches complémentaires :

1) **Modèle économétrique (principal)** : estimation de l’effet moyen des médailles sur la **croissance** des licenciés, avec **effets fixes sport** et **effets fixes année**, et erreurs robustes **clusterisées par sport**.

2) **Diagnostics complémentaires (“preuves”)** : modèles prédictifs simples pour quantifier le rôle de l’inertie (niveau passé) vs les médailles.
Ces diagnostics sont descriptifs/prédictifs, pas causaux.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# --- package
from model.feature_engineering import (
    build_lic_features,
    ajouter_jo_ref,
    build_df_sport,
    build_df_med_long,
    merge_medals,
)
from model.modele_econo import fit_modele_croissance
from model.modele_predictif import train_ridge_predictif, plot_pred_sport


## Construction du dataset final (panel sport–année)

Étapes :
1) agrégation / features sur la base licences brute (niveau sport–année) ;
2) ajout de `jo_ref` (JO de référence associé à chaque année) ;
3) transformation des médailles en format long (2016/2020/2024 -> lignes `jo_ref`) ;
4) merge des médailles dans le panel (sport, année, jo_ref).

On obtient `df_model_full` au niveau (code_sport, annee), avec `nb_licencies` et `total_medailles` (qui correspond bien aux médailles du dernier JO de référence).


In [ ]:
# 1) features licences (sport-année) + lags/croissances si possible
df_agg = build_lic_features(df_lic)

# 2) jo_ref (JO associé à l'année)
df_agg = ajouter_jo_ref(df_agg, annee_col="annee")

# 3) médailles : df_med (wide) -> df_med_long (jo_ref, or/argent/bronze/total)
df_sport = build_df_sport(df_med)          # 1 ligne par sport avec colonnes 2016_*, 2020_*, 2024_*
df_med_long = build_df_med_long(df_sport)  # format long: jo_ref + total_medailles, etc.

# 4) merge médailles dans le panel
df_model_full = merge_medals(df_agg, df_med_long)

print("df_model_full shape:", df_model_full.shape)
df_model_full.head()



## Sanity checks (cohérence du panel)

On vérifie :
- unicité des observations (code_sport, annee) ;
- cohérence : pour un sport donné, `total_medailles` doit être constant à `jo_ref` donné.


In [ ]:
print("Duplicats (code_sport, annee):", df_model_full.duplicated(["code_sport", "annee"]).sum())

tmp = df_model_full.groupby(["code_sport", "jo_ref"])["total_medailles"].nunique().reset_index()
print("Max nunique(total_medailles) par (sport, jo_ref):", tmp["total_medailles"].max())

df_model_full.groupby(["annee", "jo_ref"]).size().head(20)


## Modèle économétrique principal : effet des médailles sur la croissance (panel FE)

Spécification estimée :

\[
\Delta \log(1+Lic_{s,t}) = \beta \cdot Med_{s,JO} + \alpha_s + \gamma_t + \varepsilon_{s,t}
\]

- \(\Delta \log(1+Lic)\) : diff log (approx croissance annuelle)
- \(\alpha_s\) : **effets fixes sport** → contrôle la popularité structurelle (sports “gros” vs “petits”)
- \(\gamma_t\) : **effets fixes année** → contrôle les chocs communs (COVID, tendances générales)
- Erreurs standards **cluster sport** : autorise corrélation des erreurs dans un même sport au cours du temps.

Interprétation : si \(\beta\) est petit, \(100\beta\) ≈ impact en points de % sur la croissance annuelle associée à +1 médaille.


In [ ]:
model_econo = fit_modele_croissance(
    df_model_full=df_model_full,
    start_year=2017,
    end_year=2024,
    controls=["part_femmes", "age_mean", "nb_departements_actifs"],  # garde seulement si dispo
    medals_col="total_medailles",
)

print(model_econo.summary())

In [ ]:
beta = float(model_econo.params["med_last"])
ci = model_econo.conf_int().loc["med_last"]

print(f"beta(médailles) = {beta:.6f}")
print(f"Interprétation (approx): +1 médaille -> {100*beta:.2f}% de croissance annuelle")
print(f"IC 95%: [{100*ci[0]:.2f}%, {100*ci[1]:.2f}%]")


### Discussion / limites (à dire explicitement)

Même avec effets fixes sport et année, l’interprétation causale doit rester prudente :
- **endogénéité possible** : un sport “fort” (investissements, infrastructures) peut à la fois gagner des médailles et attirer des licenciés ;
- **hétérogénéité** : l’effet peut varier fortement selon les sports (médiatisation, accessibilité, clubs, etc.) ;
- le modèle identifie un **effet moyen** (toutes disciplines confondues), pas un effet sport-par-sport strictement causal.


## Diagnostics (“preuves”) : inertie vs médailles

Objectif : montrer empiriquement que le nombre de licenciés est très “persistant” (inertie),
et que les médailles apportent une information plus marginale en prédiction.

Ces résultats sont prédictifs/descriptifs : ils ne sont pas des preuves causales.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

def eval_ablation_models(df_model_full, train_years=(2017, 2020), test_years=(2022, 2024),
                         medals_col="total_medailles", alpha=1.0):
    df = df_model_full.sort_values(["code_sport","annee"]).copy()
    df["y"] = np.log1p(df["nb_licencies"])
    df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
    df["med_last"] = df[medals_col]
    df = df.dropna(subset=["log_lag1"]).copy()

    train_df = df[df["annee"].between(*train_years)].copy()
    test_df  = df[df["annee"].between(*test_years)].copy()

    def fit_predict(feats):
        Xtr = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
        Xte = pd.get_dummies(test_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
        Xte = Xte.reindex(columns=Xtr.columns, fill_value=0).astype("float32")

        ytr = train_df["y"].astype("float32").values
        yte = test_df["y"].astype("float32").values

        m = Ridge(alpha=alpha)
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)

        return float(r2_score(yte, pred)), float(mean_absolute_error(np.expm1(yte), np.expm1(pred)))

    rows = []
    for name, feats in [
        ("M0: inertie seule (lag)", ["log_lag1"]),
        ("M1: médailles seules", ["med_last"]),
        ("M2: inertie + médailles", ["log_lag1","med_last"]),
    ]:
        r2, mae = fit_predict(feats)
        rows.append({"modele": name, "R2_log": r2, "MAE_niveau": mae})
    return pd.DataFrame(rows)

ablation = eval_ablation_models(df_model_full)
ablation


**Interprétation attendue** :
- Si M0 est déjà très bon → forte inertie des licenciés.
- Si M2 n’améliore que faiblement M0 → les médailles ont un apport prédictif marginal sur cette fenêtre.
- Si M1 est “pas si nul”, cela peut refléter une corrélation structurelle (sports forts = médailles + licenciés).


In [ ]:
def ridge_coefficients_simple(df_model_full, train_years=(2017, 2020), medals_col="total_medailles", alpha=1.0):
    df = df_model_full.sort_values(["code_sport","annee"]).copy()
    df["y"] = np.log1p(df["nb_licencies"])
    df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
    df["trend"] = df.groupby("code_sport")["annee"].transform(lambda s: s - s.min())
    df["med_last"] = df[medals_col]
    df = df.dropna(subset=["log_lag1"]).copy()

    train_df = df[df["annee"].between(*train_years)].copy()
    feats = ["log_lag1", "trend", "med_last"]

    X = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
    y = train_df["y"].astype("float32").values

    m = Ridge(alpha=alpha)
    m.fit(X, y)

    coef = pd.Series(m.coef_, index=X.columns).sort_values(key=lambda s: s.abs(), ascending=False)
    return coef

coef = ridge_coefficients_simple(df_model_full)
coef.head(15)


Si `log_lag1` domine nettement, cela confirme empiriquement que le niveau passé de licenciés explique l’essentiel du niveau courant (inertie).


## Modèle prédictif secondaire (outil de visualisation, pas causal)

On utilise le modèle Ridge de notre package pour produire :
- une performance globale (R² en log et MAE en niveau),
- des courbes observé vs prédit sur la période test, sport par sport.

Ce modèle est surtout utilisé pour illustrer la dynamique (inertie) et visualiser les trajectoires.


In [ ]:
bundle = train_ridge_predictif(
    df_model_full=df_model_full,
    train_years=(2017, 2020),
    test_years=(2022, 2024),
    medals_col="total_medailles",
    alpha=1.0,
    extra_controls=None,   # auto: part_femmes, age_mean, nb_departements_actifs si dispo
)

bundle["metrics"]


In [ ]:
_ = plot_pred_sport(bundle, "HAN")  # hand
_ = plot_pred_sport(bundle, "ATH")  # athlé


## Conclusion (modélisation)

- Le modèle économétrique (diff-log + FE sport + FE année + SE cluster sport) fournit une estimation interprétable de l’association entre médailles et croissance des licenciés.
- Les diagnostics montrent que la dynamique des licenciés est fortement inertielle (le passé explique une grande partie du présent), ce qui limite mécaniquement la capacité des médailles à expliquer la totalité des variations.
- Le modèle prédictif sert surtout à visualiser les trajectoires sport par sport et à appuyer le diagnostic d’inertie ; il ne doit pas être interprété comme une estimation causale.


In [ ]:
#### a mettre dans le package #### 
def simulate_medal_shock(bundle, code_sport: str, delta_medals: int):
    """
    Simulation contrefactuelle simple (NON causale) :
    Que se passerait-il dans le modèle prédictif si un sport avait
    gagné +delta_medals médailles au dernier JO ?

    Paramètres
    ----------
    bundle : dict
        Sortie de train_ridge_predictif (contient model, X_train, features, test_df, etc.)
    code_sport : str
        Code sport (ex: "HAN", "ATH")
    delta_medals : int
        Choc artificiel sur le nombre de médailles (+/-)

    Retour
    ------
    DataFrame avec observé / prédit baseline / prédit avec choc
    """

    model = bundle["model"]
    test_df = bundle["test_df"].copy()
    features_num = bundle["features_num"]
    X_train_cols = bundle["X_train_cols"]

    # garder uniquement le sport demandé
    d = test_df[test_df["code_sport"] == code_sport].sort_values("annee").copy()
    if d.empty:
        print(f"Aucune observation test pour le sport {code_sport}")
        return None

    # baseline (sans choc)
    X_base = pd.get_dummies(
        d[["code_sport"] + features_num],
        columns=["code_sport"],
        drop_first=True
    ).astype("float32")
    X_base = X_base.reindex(columns=X_train_cols, fill_value=0).astype("float32")

    d["pred_log_base"] = model.predict(X_base)
    d["pred_nb_base"] = np.expm1(d["pred_log_base"])

    # scénario contrefactuel : + delta_medals
    d_shock = d.copy()
    if "med_last" not in d_shock.columns:
        raise ValueError("La colonne 'med_last' est absente du dataset.")

    d_shock["med_last"] = d_shock["med_last"] + delta_medals

    X_shock = pd.get_dummies(
        d_shock[["code_sport"] + features_num],
        columns=["code_sport"],
        drop_first=True
    ).astype("float32")
    X_shock = X_shock.reindex(columns=X_train_cols, fill_value=0).astype("float32")

    d["pred_log_shock"] = model.predict(X_shock)
    d["pred_nb_shock"] = np.expm1(d["pred_log_shock"])

    # graphique
    plt.figure(figsize=(9,5))
    plt.plot(d["annee"], d["nb_licencies"], marker="o", linewidth=2, label="observé")
    plt.plot(d["annee"], d["pred_nb_base"], marker="o", linestyle="--", label="prédit (baseline)")
    plt.plot(d["annee"], d["pred_nb_shock"], marker="o", linestyle=":", label=f"prédit (+{delta_medals} médailles)")

    plt.title(f"{code_sport} — Simulation prédictive (choc de médailles)")
    plt.xlabel("Année")
    plt.ylabel("Nombre de licenciés")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return d[[
        "code_sport", "annee", "nb_licencies",
        "pred_nb_base", "pred_nb_shock"
    ]]


In [ ]:
simulate_medal_shock(bundle, code_sport="HAN", delta_medals=3)

Cette simulation n'a pas voaction à établir un effet causal des médailles. Elle repose sur un modele prédictif intégrat une forte inertie (niveau passé des licenciés). Elle permet uniquement d'illustrer l'ordre de grandeur de la variation prédte pa rle modèle lorsque l'on modifie ex post le nombre de médailles, toutes choses égales par ailleurs.